In [ ]:
import os
from PIL import Image
import supervision as sv
from tqdm import tqdm
from inference.models.yolo_world.yolo_world import YOLOWorld
import xml.etree.ElementTree as ET
from xml.dom import minidom

In [ ]:
import os
from PIL import Image
import supervision as sv
from tqdm import tqdm
from inference.models.yolo_world.yolo_world import YOLOWorld
import xml.etree.ElementTree as ET
from xml.dom import minidom

def extract_classes_from_named_xml(xml_path):
    classes = set()
    name_mapping = {
        "Container": ["Container ship"],
        "Chemical": ["Chemical ship"],
        "Pilot": ["Pilot Vessel"],
        "Passenger-RoRo": ["RoRo", "Passenger ship", "Passenger vessel"]
    }

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        for obj in root.findall("object"):
            name = obj.find("name").text
            if name:
                name = name.strip()
                mapped = name_mapping.get(name, [name])  # Dùng mapping nếu có
                for m in mapped:
                    classes.add(m)
                    print(m)
    except Exception:
        pass

    return sorted(list(classes))



def create_voc_xml(detections, file_name_str, path_str, output_path, width=640, height=640, depth=3, classes=None):
    annotation = ET.Element("annotation")
    ET.SubElement(annotation, "folder").text = ""
    ET.SubElement(annotation, "filename").text = file_name_str
    ET.SubElement(annotation, "path").text = os.path.join(path_str, file_name_str)

    source = ET.SubElement(annotation, "source")
    ET.SubElement(source, "database").text = "roboflow.com"

    size = ET.SubElement(annotation, "size")
    ET.SubElement(size, "width").text = str(width)
    ET.SubElement(size, "height").text = str(height)
    ET.SubElement(size, "depth").text = str(depth)

    ET.SubElement(annotation, "segmented").text = "0"

    for (x1, y1, x2, y2), class_id in zip(detections.xyxy, detections.class_id):
        class_name = classes[class_id] if classes and class_id < len(classes) else "object"
        obj = ET.SubElement(annotation, "object")
        ET.SubElement(obj, "name").text = class_name
        ET.SubElement(obj, "pose").text = "Unspecified"
        ET.SubElement(obj, "truncated").text = "0"
        ET.SubElement(obj, "difficult").text = "0"
        ET.SubElement(obj, "occluded").text = "0"

        bndbox = ET.SubElement(obj, "bndbox")
        ET.SubElement(bndbox, "xmin").text = str(int(x1))
        ET.SubElement(bndbox, "xmax").text = str(int(x2))
        ET.SubElement(bndbox, "ymin").text = str(int(y1))
        ET.SubElement(bndbox, "ymax").text = str(int(y2))

    ET.SubElement(annotation, "metadata").text = " "
    rough_string = ET.tostring(annotation, 'utf-8')
    reparsed = minidom.parseString(rough_string)
    pretty_xml = reparsed.toprettyxml(indent="  ")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(pretty_xml)

# ---------------------------
# Cấu hình đường dẫn
# ---------------------------

#image_dir = '/home/aiplatform/projects/test/output_ships'
subfolder_name = "img_cameras_2023-10-13-14-19-51_138_jpg.rf.01fc8a9316fbce07e5bbaf031562b442"  # thay bằng tên thật
image_dir = os.path.join("/home/aiplatform/projects/test/data/VESSELimg/valid_Flux", subfolder_name)

base_name = os.path.basename(image_dir.rstrip("/"))
xml_dir = "/home/aiplatform/projects/test/data/VESSELimg/VOC_valid"
xml_file_path = os.path.join(xml_dir, base_name + ".xml")

# Lấy danh sách class từ file XML
classes = extract_classes_from_named_xml(xml_file_path)
#classes=['container vessel',  'tugboat vessel', 'pilot ship', 'buoy']
# Load model với danh sách class
model = YOLOWorld(model_id="yolo_world/l")
model.set_classes(classes)

# Xử lý từng ảnh
for file_name in tqdm(os.listdir(image_dir)):
    if file_name.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
        image_path = os.path.join(image_dir, file_name)

        try:
            image = Image.open(image_path).convert("RGB")
        except Exception:
            continue

        width, height = image.size
        results = model.infer(image)
        detections = sv.Detections.from_inference(results)

        mask = detections.confidence > 0.05
        detections.xyxy = detections.xyxy[mask]
        detections.class_id = detections.class_id[mask]
        detections.confidence = detections.confidence[mask]

        if len(detections.xyxy) == 0:
            #os.remove(image_path)
            continue

        xml_name = os.path.splitext(file_name)[0] + ".xml"
        output_path = os.path.join(image_dir, xml_name)

        create_voc_xml(detections, file_name, image_dir, output_path, width, height, classes=classes)


In [4]:
import os
from PIL import Image
import supervision as sv
from tqdm import tqdm
from inference.models.yolo_world.yolo_world import YOLOWorld
import xml.etree.ElementTree as ET
from xml.dom import minidom


def create_voc_xml(detections, file_name_str, path_str, output_path, width=640, height=640, depth=3, classes=None):
    annotation = ET.Element("annotation")
    ET.SubElement(annotation, "folder").text = ""
    ET.SubElement(annotation, "filename").text = file_name_str
    ET.SubElement(annotation, "path").text = os.path.join(path_str, file_name_str)

    source = ET.SubElement(annotation, "source")
    ET.SubElement(source, "database").text = "roboflow.com"

    size = ET.SubElement(annotation, "size")
    ET.SubElement(size, "width").text = str(width)
    ET.SubElement(size, "height").text = str(height)
    ET.SubElement(size, "depth").text = str(depth)

    ET.SubElement(annotation, "segmented").text = "0"

    for (x1, y1, x2, y2), class_id in zip(detections.xyxy, detections.class_id):
        class_name = classes[class_id] if classes and class_id < len(classes) else "object"
        obj = ET.SubElement(annotation, "object")
        ET.SubElement(obj, "name").text = class_name
        ET.SubElement(obj, "pose").text = "Unspecified"
        ET.SubElement(obj, "truncated").text = "0"
        ET.SubElement(obj, "difficult").text = "0"
        ET.SubElement(obj, "occluded").text = "0"

        bndbox = ET.SubElement(obj, "bndbox")
        ET.SubElement(bndbox, "xmin").text = str(int(x1))
        ET.SubElement(bndbox, "xmax").text = str(int(x2))
        ET.SubElement(bndbox, "ymin").text = str(int(y1))
        ET.SubElement(bndbox, "ymax").text = str(int(y2))

    ET.SubElement(annotation, "metadata").text = " "
    rough_string = ET.tostring(annotation, 'utf-8')
    reparsed = minidom.parseString(rough_string)
    pretty_xml = reparsed.toprettyxml(indent="  ")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(pretty_xml)


# ---------------------------
# Cấu hình đường dẫn
# ---------------------------
image_root_dir = '/home/aiplatform/projects/test/data/VESSELimg/Fluxgenerated_Pilot_1_merge'
classes = ['ship', 'boat','vessel']

# Load model
model = YOLOWorld(model_id="yolo_world/l")
model.set_classes(classes)

# Duyệt tất cả các thư mục con
for root, dirs, files in os.walk(image_root_dir):
    for file_name in tqdm(files, desc=f"Processing in {root}"):
        if file_name.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
            image_path = os.path.join(root, file_name)

            try:
                image = Image.open(image_path).convert("RGB")
            except Exception:
                continue

            width, height = image.size
            results = model.infer(image)
            detections = sv.Detections.from_inference(results)

            mask = detections.confidence > 0.05
            detections.xyxy = detections.xyxy[mask]
            detections.class_id = detections.class_id[mask]
            detections.confidence = detections.confidence[mask]

            if len(detections.xyxy) == 0:
                os.remove(image_path)
                continue

            xml_name = os.path.splitext(file_name)[0] + ".xml"
            output_path = os.path.join(root, xml_name)

            create_voc_xml(detections, file_name, root, output_path, width, height, classes=classes)


[05/30/25 10:21:00] WARNING  Your inference package version 0.49.2 is out of date! Please upgrade to ]8;id=620909;file:///opt/conda/miniconda/lib/python3.10/site-packages/inference/core/__init__.py\__init__.py]8;;\:]8;id=226797;file:///opt/conda/miniconda/lib/python3.10/site-packages/inference/core/__init__.py#41\41]8;;\
                             version 0.50.2 of inference for the latest features and bug fixes by                  
                             running `pip install --upgrade inference`.                                            

Creating inference sessions


CLIP model loaded in 6.24 seconds


Processing in /home/aiplatform/projects/test/data/VESSELimg/Fluxgenerated_Pilot_1_merge: 0it [00:00, ?it/s]
Processing in /home/aiplatform/projects/test/data/VESSELimg/Fluxgenerated_Pilot_1_merge/train1_Flux_group1: 100%|██████████| 600/600 [00:44<00:00, 13.34it/s]
Processing in /home/aiplatform/projects/test/data/VESSELimg/Fluxgenerated_Pilot_1_merge/train1_Flux_group2: 100%|██████████| 600/600 [00:18<00:00, 32.80it/s]
Processing in /home/aiplatform/projects/test/data/VESSELimg/Fluxgenerated_Pilot_1_merge/train1_Flux_group4: 100%|██████████| 600/600 [00:18<00:00, 31.70it/s]
Processing in /home/aiplatform/projects/test/data/VESSELimg/Fluxgenerated_Pilot_1_merge/train1_Flux_group3: 100%|██████████| 600/600 [00:18<00:00, 32.53it/s]
Processing in /home/aiplatform/projects/test/data/VESSELimg/Fluxgenerated_Pilot_1_merge/train1_Flux_group5: 100%|██████████| 600/600 [00:18<00:00, 32.58it/s]


In [ ]:
!pip install zip

In [ ]:
import cv2

In [2]:
import os
from PIL import Image
import supervision as sv
from tqdm import tqdm
from inference.models.yolo_world.yolo_world import YOLOWorld
import xml.etree.ElementTree as ET
from xml.dom import minidom


def create_voc_xml(detections, file_name_str, path_str, output_path, width=640, height=640, depth=3, classes=None):
    annotation = ET.Element("annotation")
    ET.SubElement(annotation, "folder").text = ""
    ET.SubElement(annotation, "filename").text = file_name_str
    ET.SubElement(annotation, "path").text = os.path.join(path_str, file_name_str)

    source = ET.SubElement(annotation, "source")
    ET.SubElement(source, "database").text = "roboflow.com"

    size = ET.SubElement(annotation, "size")
    ET.SubElement(size, "width").text = str(width)
    ET.SubElement(size, "height").text = str(height)
    ET.SubElement(size, "depth").text = str(depth)

    ET.SubElement(annotation, "segmented").text = "0"

    for (x1, y1, x2, y2), class_id in zip(detections.xyxy, detections.class_id):
        class_name = classes[class_id] if classes and class_id < len(classes) else "object"
        obj = ET.SubElement(annotation, "object")
        ET.SubElement(obj, "name").text = class_name
        ET.SubElement(obj, "pose").text = "Unspecified"
        ET.SubElement(obj, "truncated").text = "0"
        ET.SubElement(obj, "difficult").text = "0"
        ET.SubElement(obj, "occluded").text = "0"

        bndbox = ET.SubElement(obj, "bndbox")
        ET.SubElement(bndbox, "xmin").text = str(int(x1))
        ET.SubElement(bndbox, "xmax").text = str(int(x2))
        ET.SubElement(bndbox, "ymin").text = str(int(y1))
        ET.SubElement(bndbox, "ymax").text = str(int(y2))

    ET.SubElement(annotation, "metadata").text = " "
    rough_string = ET.tostring(annotation, 'utf-8')
    reparsed = minidom.parseString(rough_string)
    pretty_xml = reparsed.toprettyxml(indent="  ")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(pretty_xml)


# ---------------------------
# Cấu hình đường dẫn
# ---------------------------
image_root_dir = '/home/aiplatform/projects/test/data/VESSELimg/Fluxgenerated_Pilot_Test_merge/train1_Flux_group1'
classes = ['ship','boat']

# Load model
model = YOLOWorld(model_id="yolo_world/l")
model.set_classes(classes)

# Lấy danh sách file ảnh trong thư mục gốc (không đệ quy)
image_files = [f for f in os.listdir(image_root_dir)
               if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]

for file_name in tqdm(image_files, desc="Processing images"):
    image_path = os.path.join(image_root_dir, file_name)

    try:
        image = Image.open(image_path).convert("RGB")
    except Exception:
        continue

    width, height = image.size
    results = model.infer(image)
    detections = sv.Detections.from_inference(results)

    mask = detections.confidence > 0.05
    detections.xyxy = detections.xyxy[mask]
    detections.class_id = detections.class_id[mask]
    detections.confidence = detections.confidence[mask]

    if len(detections.xyxy) == 0:
        os.remove(image_path)
        continue

    xml_name = os.path.splitext(file_name)[0] + ".xml"
    output_path = os.path.join(image_root_dir, xml_name)

    create_voc_xml(detections, file_name, image_root_dir, output_path, width, height, classes=classes)


[05/28/25 13:12:18] WARNING  Your inference package version 0.49.2 is out of date! Please upgrade to ]8;id=65094;file:///opt/conda/miniconda/lib/python3.10/site-packages/inference/core/__init__.py\__init__.py]8;;\:]8;id=422104;file:///opt/conda/miniconda/lib/python3.10/site-packages/inference/core/__init__.py#41\41]8;;\
                             version 0.50.1 of inference for the latest features and bug fixes by                  
                             running `pip install --upgrade inference`.                                            

Creating inference sessions


CLIP model loaded in 6.38 seconds


Processing images: 100%|██████████| 300/300 [00:45<00:00,  6.55it/s]


In [1]:
#Gán nhãn lại 1 thư mục
import os
import cv2
import xml.etree.ElementTree as ET
from PIL import Image
import torch
from torchvision import transforms, models

from torchvision import datasets, transforms

# Đường dẫn tới thư mục chứa ảnh con (dùng lúc huấn luyện)
train_data_dir = "/home/aiplatform/projects/test/data/VESSELimg/classification"

# Áp dụng transform đơn giản vì ta chỉ cần danh sách class
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
dataset = datasets.ImageFolder(train_data_dir, transform=transform)

# Lấy danh sách class
class_names = dataset.classes
print(class_names)


# === Cấu hình ===
data_dir =  '/home/aiplatform/projects/test/data/VESSELimg/Fluxgenerated_Pilot_Test_merge/train1_Flux_group1'
model_path = "best_model_mobilenet.pth"  # đường dẫn tới mô hình đã huấn luyện
#class_names = ['Container', 'Tugboat']  # danh sách class đúng theo lúc huấn luyện

# === Load mô hình ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = torch.nn.Linear(model.last_channel, len(class_names))
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# === Transform ảnh cắt ===
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# === Duyệt file XML ===
for filename in os.listdir(data_dir):
    if not filename.endswith(".xml"):
        continue

    xml_path = os.path.join(data_dir, filename)
    tree = ET.parse(xml_path)
    root = tree.getroot()

    image_name = root.find('filename').text
    image_path = os.path.join(data_dir, image_name)
    if not os.path.exists(image_path):
        print(f"Không tìm thấy ảnh: {image_path}")
        continue

    image = cv2.imread(image_path)
    objects = root.findall('object')

    for obj in objects:
        name_tag = obj.find('name')
         # Chỉ xử lý object có tên "ship"

        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)

        # Cắt object từ ảnh gốc
        cropped = image[ymin:ymax, xmin:xmax]
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(cropped_rgb)

        # Chuyển đổi và phân loại
        input_tensor = transform(pil_img).unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(input_tensor)
            pred_idx = torch.argmax(output, dim=1).item()
            pred_class = class_names[pred_idx]

        # Cập nhật tên object trong XML
        name_tag.text = pred_class

    # Ghi đè lại file XML
    tree.write(xml_path, encoding='utf-8')
    #print(f"✅ Đã cập nhật: {filename}")

print("🎉 Đã xử lý xong tất cả file XML.")


['Buoy', 'Chemical', 'Container', 'Passenger-RoRo', 'Pilot', 'Tugboat']


/opt/conda/miniconda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/miniconda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


🎉 Đã xử lý xong tất cả file XML.


In [5]:
#Gán nhãn lại tất cả các thư mục
import os
import cv2
import xml.etree.ElementTree as ET
from PIL import Image
import torch
from torchvision import transforms, models

from torchvision import datasets, transforms

# Đường dẫn tới thư mục chứa ảnh con (dùng lúc huấn luyện)
train_data_dir = "/home/aiplatform/projects/test/data/VESSELimg/classification"

# Áp dụng transform đơn giản vì ta chỉ cần danh sách class
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
dataset = datasets.ImageFolder(train_data_dir, transform=transform)

# Lấy danh sách class
class_names = dataset.classes
print(class_names)


# === Cấu hình ===
data_dir = "/home/aiplatform/projects/test/data/VESSELimg/Fluxgenerated_Pilot_1_merge"
model_path = "best_model_mobilenet.pth"  # đường dẫn tới mô hình đã huấn luyện
#class_names = ['Container', 'Tugboat']  # danh sách class đúng theo lúc huấn luyện

# === Load mô hình ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = torch.nn.Linear(model.last_channel, len(class_names))
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# === Transform ảnh cắt ===
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# === Duyệt file XML ===
# === Duyệt file XML trong thư mục và các thư mục con ===
for root_dir, dirs, files in os.walk(data_dir):
    for filename in files:
        if not filename.endswith(".xml"):
            continue

        xml_path = os.path.join(root_dir, filename)
        tree = ET.parse(xml_path)
        root = tree.getroot()

        image_name = root.find('filename').text
        image_path = os.path.join(root_dir, image_name)
        if not os.path.exists(image_path):
            print(f"Không tìm thấy ảnh: {image_path}")
            continue

        image = cv2.imread(image_path)
        objects = root.findall('object')

        for obj in objects:
            name_tag = obj.find('name')            
            bbox = obj.find('bndbox')
            xmin = int(bbox.find('xmin').text)
            ymin = int(bbox.find('ymin').text)
            xmax = int(bbox.find('xmax').text)
            ymax = int(bbox.find('ymax').text)

            # Cắt object từ ảnh gốc
            cropped = image[ymin:ymax, xmin:xmax]
            cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(cropped_rgb)

            # Chuyển đổi và phân loại
            input_tensor = transform(pil_img).unsqueeze(0).to(device)
            with torch.no_grad():
                output = model(input_tensor)
                pred_idx = torch.argmax(output, dim=1).item()
                pred_class = class_names[pred_idx]

            # Cập nhật tên object trong XML
            name_tag.text = pred_class

        # Ghi đè lại file XML
        tree.write(xml_path, encoding='utf-8')
        #print(f"✅ Đã cập nhật: {xml_path}")

print("🎉 Đã xử lý xong tất cả file XML.")


['Buoy', 'Chemical', 'Container', 'Passenger-RoRo', 'Pilot', 'Tugboat']


🎉 Đã xử lý xong tất cả file XML.
